In [1]:
from ks_models import models 
import pandas as pd
import numpy as np

In [2]:
X = pd.read_csv('data/archive/fighter_details.csv')
X.head()

,id,name,nick_name,wins,losses,draws,height,weight,reach,stance,dob,splm,str_acc,sapm,str_def,td_avg,td_avg_acc,td_def,sub_avg
0,8f382b3baa954d2a,Jessica Aguilar,Jag,20,8,0,160.02,52.16,160.02,Orthodox,"May 08, 1982",4.93,50,7.19,53,0.94,25,50,0.2
1,483a953b18d73bb3,Deron Winn,NaN,7,3,0,167.64,83.91,177.80,Orthodox,"Jun 13, 1989",4.55,44,6.21,46,4.28,52,40,0.0
2,232c582f29f8f65e,Gegard Mousasi,NaN,42,6,2,187.96,83.91,193.04,Orthodox,"Aug 01, 1985",3.75,50,1.21,68,1.59,60,59,1.1
3,236a37d96d476164,Mike Pierce,NaN,17,7,0,172.72,77.11,180.34,Orthodox,"Sep 01, 1980",2.62,42,2.36,62,3.08,42,71,0.2
4,203c957eac95dd87,Hyun Gyu Lim,The Ace,13,7,1,190.50,77.11,195.58,Orthodox,"Jan 16, 1985",4.45,41,5.07,51,0.23,100,73,0.2


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

y = X["wins"] / (X["wins"] + X["losses"])
X['stance'] = X['stance'].where(X['stance'] == 'Orthodox', 'Southpaw')

drop = ['name', 'id', 'nick_name', 'dob']
numerical_features = [col for col in X.select_dtypes(include=['number']).columns if col not in drop]
categorical_features = ['stance']

numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('imputer', SimpleImputer(strategy='mean'))
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ('imputer', SimpleImputer(strategy='most_frequent'))
])  

pipe = Pipeline([
    ('transformers', ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='drop'
    )),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_train)
X_train_transformed = pd.DataFrame(pipe.transform(X_train))
X_test_transformed = pd.DataFrame(pipe.transform(X_test))

X_train_transformed.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,-0.257240,-0.263818,-0.318918,0.776911,0.392515,8.893127e-16,-0.165501,0.819127,-0.578670,-0.009086,3.413230,1.438954,-1.713579,1.099951,1.0,0.0
1,0.041048,-0.263818,-0.318918,1.060780,0.392515,1.013748e+00,0.062418,-0.203068,0.075233,-0.073394,0.263945,0.314051,1.681547,-0.555459,1.0,0.0
2,-0.555527,-0.930439,-0.318918,-1.210170,-0.858278,-3.959273e-01,-0.153812,-0.271215,0.600499,-0.137702,1.161826,0.113175,1.681547,-0.095623,1.0,0.0
3,-0.953244,-0.708232,-0.318918,0.209174,0.017497,7.396443e-02,0.892275,0.682835,0.616579,0.119531,0.391257,-0.087700,1.681547,-0.555459,1.0,0.0
4,3.023925,3.735909,0.954314,-0.358564,-0.357520,7.396443e-02,-0.896008,-0.271215,-0.600109,0.698306,0.203640,0.394401,-0.864798,0.456180,0.0,1.0


In [4]:
LinearRegression = models.LinearRegression()
LinearRegression.fit(X_train_transformed, y_train)
LinearRegression.predict(X_test_transformed)
print(LinearRegression.score(X_train_transformed, y_train))
print(LinearRegression.score(X_test_transformed, y_test))



0.6050561672541048
0.6135896229886929


In [5]:
from sklearn.linear_model import LinearRegression
LinearRegression = LinearRegression()
LinearRegression.fit(X_train_transformed, y_train)
LinearRegression.predict(X_test_transformed)
print(LinearRegression.score(X_train_transformed, y_train))
print(LinearRegression.score(X_test_transformed, y_test))

0.6050561672541048
0.6134200713365732


In [6]:
from sklearn.model_selection import cross_validate, LeaveOneOut
from sklearn.linear_model import LinearRegression
import numpy as np

LinearRegression_sk = LinearRegression()
cv_results = cross_validate(
    LinearRegression_sk, 
    X_train_transformed, 
    y_train,
    cv=LeaveOneOut(),
    scoring='neg_mean_squared_error',  # Use MSE instead of R²
    return_train_score=True
)
print(f"\nLOOCV with MSE:")
print(f"Train MSE: {-cv_results['train_score'].mean():.4f}")
print(f"Test MSE: {-cv_results['test_score'].mean():.4f}")
print(f"Test RMSE: {np.sqrt(-cv_results['test_score'].mean()):.4f}")



LOOCV with MSE:
Train MSE: 0.0093
Test MSE: 0.0102
Test RMSE: 0.1009


In [7]:
from ks_models.learning_utils import utils
from ks_models.models import LinearRegression

LinearRegression = LinearRegression()
print(utils.loocv(LinearRegression, X_train_transformed, y_train))



0.010185489963161732


In [8]:
from ks_models import models
RidgeRegression = models.RidgeRegression(1.0)
RidgeRegression.fit(X_train_transformed, y_train)
RidgeRegression.predict(X_test_transformed)
print(RidgeRegression.score(X_train_transformed, y_train))
print(RidgeRegression.score(X_test_transformed, y_test))



0.6050526120111914
0.6135297700797919


In [9]:
from sklearn.linear_model import Ridge
RidgeRegression = Ridge(alpha = 1.0)
RidgeRegression.fit(X_train_transformed, y_train)
RidgeRegression.predict(X_test_transformed)
print(RidgeRegression.score(X_train_transformed, y_train))
print(RidgeRegression.score(X_test_transformed, y_test))


0.6050553455938951
0.6133247889577966


In [10]:
# Load UFC dataset and replicate feature engineering from eda.ipynb
from pathlib import Path

data_path = Path("data") / "archive" / "UFC.csv"
columns_to_keep = [
    'r_id', 'b_id', 'r_name', 'b_name', 'date', 'method',
    'r_wins', 'b_wins', 'r_losses', 'b_losses', 'r_draws', 'b_draws',
    'b_height', 'r_height', 'b_reach', 'r_reach', 'b_stance', 'r_stance',
    'r_weight', 'b_weight', 'r_dob', 'b_dob', 'r_splm', 'b_splm',
    'r_td_acc', 'b_td_acc', 'r_str_acc', 'b_str_acc', 'r_sapm', 'b_sapm',
    'r_str_def', 'b_str_def', 'r_td_avg', 'b_td_avg', 'r_td_avg_acc',
    'b_td_avg_acc', 'r_td_def', 'b_td_def', 'r_sub_avg', 'b_sub_avg',
    'division', 'title_fight', 'winner_id'
]

ufc_df = pd.read_csv(data_path)[columns_to_keep]

# Parse dates
ufc_df['r_dob'] = pd.to_datetime(ufc_df['r_dob'], errors='coerce')
ufc_df['b_dob'] = pd.to_datetime(ufc_df['b_dob'], errors='coerce')
ufc_df['date'] = pd.to_datetime(ufc_df['date'], errors='coerce')

# Winner flag (True if red fighter won)
ufc_df['winner'] = ufc_df['winner_id'] == ufc_df['r_id']

# Ages
today = pd.Timestamp.today()
ufc_df['r_age'] = ((today - ufc_df['r_dob']).dt.days / 365.25).round(2)
ufc_df['b_age'] = ((today - ufc_df['b_dob']).dt.days / 365.25).round(2)

# Win-loss ratios (avoid div by zero)
r_total = ufc_df['r_wins'] + ufc_df['r_losses']
b_total = ufc_df['b_wins'] + ufc_df['b_losses']
ufc_df['r_win_loss_ratio'] = ufc_df['r_wins'] / r_total.replace({0: np.nan})
ufc_df['b_win_loss_ratio'] = ufc_df['b_wins'] / b_total.replace({0: np.nan})

# Normalize stances
valid_stances = {"Orthodox", "Southpaw"}
ufc_df['r_stance'] = ufc_df['r_stance'].where(ufc_df['r_stance'].isin(valid_stances), 'Orthodox')
ufc_df['b_stance'] = ufc_df['b_stance'].where(ufc_df['b_stance'].isin(valid_stances), 'Orthodox')

# Drop helper columns not used for modeling
ufc_df = ufc_df.drop(columns=[
    'winner_id', 'division', 'r_dob', 'b_dob', 'r_id', 'b_id',
    'r_wins', 'r_losses', 'b_wins', 'b_losses', 'r_draws', 'b_draws',
    'r_name', 'b_name', 'date', 'method'
])

ufc_df.head()


,b_height,r_height,b_reach,r_reach,b_stance,r_stance,r_weight,b_weight,r_splm,b_splm,...,r_td_def,b_td_def,r_sub_avg,b_sub_avg,title_fight,winner,r_age,b_age,r_win_loss_ratio,b_win_loss_ratio
0,180.34,180.34,190.50,185.42,Orthodox,Southpaw,70.31,70.31,5.05,3.84,...,70,84,1.6,0.0,0,True,29.99,29.49,0.833333,0.857143
1,185.42,190.50,190.50,190.50,Southpaw,Orthodox,83.91,83.91,4.28,3.44,...,81,76,1.0,0.4,0,True,29.79,32.91,0.809524,0.894737
2,190.50,190.50,193.04,193.04,Orthodox,Orthodox,92.99,92.99,3.32,2.50,...,80,35,0.1,1.3,0,True,31.84,38.05,0.760000,0.629630
3,177.80,177.80,187.96,185.42,Orthodox,Orthodox,70.31,70.31,6.59,5.71,...,66,81,0.0,0.2,0,False,30.08,30.64,0.769231,0.894737
4,187.96,187.96,198.12,187.96,Orthodox,Southpaw,77.11,77.11,3.74,4.71,...,0,47,0.0,0.0,0,True,28.24,30.26,1.000000,0.666667


In [11]:
# Train/test split and preprocessing (mirrors eda.ipynb setup)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

X = ufc_df.drop(columns=['winner'])
y = ufc_df['winner']

categorical_features = ['title_fight', 'r_stance', 'b_stance']
numerical_features = [col for col in X.columns if col not in categorical_features]

numerical_transformer = make_pipeline(
    SimpleImputer(strategy='mean'),
    StandardScaler()
)

categorical_transformer = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore', sparse_output=False)
)

preprocessor = Pipeline([
    ('transform', ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_features),
            ('cat', categorical_transformer, categorical_features),
        ],
        remainder='drop'
    ))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

X_train_processed.shape, X_test_processed.shape


((6669, 34), (1668, 34))

In [12]:
# Example: test your ks_models logistic regression (replace with your function signature)
LogisticRegression = models.LogisticRegression()
LogisticRegression.fit(X_train_processed, y_train)
LogisticRegression.predict(X_test_processed)
print(LogisticRegression.score(X_train_processed, y_train))
print(LogisticRegression.score(X_test_processed, y_test))


0.7321937321937322
0.7547961630695443


In [13]:
from sklearn.linear_model import LogisticRegression
LogisticRegression = LogisticRegression()
LogisticRegression.fit(X_train_processed, y_train)
LogisticRegression.predict(X_test_processed)
print(LogisticRegression.score(X_train_processed, y_train))
print(LogisticRegression.score(X_test_processed, y_test))


0.7312940470835207
0.7571942446043165
